In [ ]:
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.item_representation.content_embeddings import (
    AmazonItemTextBuilder,
    ContentEmbeddingBuilder,
    HashingTextEmbedder,
)
from pathlib import Path

from flowrec.item_representation.content_embeddings import ItemEmbeddingStore, HFTextEmbedder
from flowrec.item_representation.semantic_id.rqvae import (
    RQVAEConfig,
    RQVAETrainConfig)
from flowrec.item_representation.semantic_id.rqvae_tokenizer import  RQVAEItemTokenizer, RQVAEItemTokenizerConfig

In [ ]:
cfg = AmazonReviewDataConfig(
    reviews_path="../data/raw/2023/Beauty_and_Personal_Care.jsonl.gz",
    metadata_path="../data/raw/2023/meta_Beauty_and_Personal_Care.jsonl.gz",
    item_key="asin",

    min_user_interactions=5,
    min_item_interactions=5,

    split_strategy="leave_one_out",
    deduplicate_consecutive = True,
    train_all_prefixes=False,

    max_history_len=20,
    batch_size=256,
    eval_batch_size=512,
)

dm = AmazonReviewDataModule(cfg)
dm.setup()

print(dm.summary())

In [ ]:
dm.item2id

In [ ]:
dm.id2item

In [ ]:
len(dm.item2id)

In [ ]:
len(dm.id2item)

In [ ]:
metadata_path = Path("../data/raw/2014/meta_Beauty.json.gz")
output_path = Path("../artifacts/item_embeddings/beauty_content_embeddings.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
text_builder = AmazonItemTextBuilder(
    include_title=True,
    include_categories=True,
    include_features=True,
    include_description=True,
    include_details=True,
    include_price=False,
    max_chars=4096,
)

embedder = HFTextEmbedder(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
    max_length=256,
    normalize=False,
    pooling="masked_mean",
    show_progress=True,
)

builder = ContentEmbeddingBuilder(
    text_builder=text_builder,
    embedder=embedder,
    item_key="parent_asin",
)

result = builder.build_from_metadata(
    metadata_path=metadata_path,
    batch_size=512,
    allowed_item_ids=list(dm.item2id.keys()),
)

result.store.save(output_path)

print("Saved embeddings to:", output_path)
print("Number of items:", len(result.store.item_ids))
print("Embedding shape:", result.store.embeddings.shape)

In [ ]:
embeddings_path = Path("../artifacts/item_embeddings/beauty_content_embeddings.pt")
output_dir = Path("../artifacts/tokenizers/rqvae_beauty")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
embedding_store = ItemEmbeddingStore.load(embeddings_path)

item_ids = embedding_store.item_ids
embeddings = embedding_store.embeddings

print("Number of items:", len(item_ids))
print("Embedding shape:", embeddings.shape)

In [ ]:
rqvae_cfg = RQVAEConfig(
    input_dim=embeddings.shape[1],
    latent_dim=32,
    hidden_dims=(512, 256, 128),
    num_codebooks=3,
    codebook_size=256,
    commitment_weight=0.25,
    dropout=0.0,
    use_batch_norm=False,
)

train_cfg = RQVAETrainConfig(
    epochs=20,
    batch_size=1024,
    learning_rate=1e-3,
    weight_decay=0.0,
    val_ratio=0.05,
    seed=42,
    gradient_clip_norm=1.0,
    device="cpu", 
    codebook_init = "kmeans"
)

tokenizer_cfg = RQVAEItemTokenizerConfig(
    rqvae=rqvae_cfg,
    train=train_cfg,

    # Recommended for text embeddings.
    normalize="none",

    # Important for TIGER-style flat token vocabulary:
    # code_l -> code_l + l * codebook_size
    level_offset_tokens=False,

    # Resolve duplicate semantic IDs by appending a suffix token.
    collision_policy="random_append",

    encode_batch_size=4096,
)


In [ ]:
tokenizer = RQVAEItemTokenizer(tokenizer_cfg)

tokenizer.fit(
    embeddings=embeddings,
    item_ids=item_ids,
)

In [ ]:
print(tokenizer.config.level_offset_tokens)
print(type(tokenizer.config.level_offset_tokens))

In [ ]:
tokenizer.save(output_dir)

semantic_ids_path = output_dir / "semantic_ids.json"

print("Saved tokenizer to:", output_dir)
print("Saved semantic IDs to:", semantic_ids_path)

In [ ]:
semantic_id_store = tokenizer.semantic_id_store
semantic_id_mapping = semantic_id_store.to_plain_dict()
semantic_id_mapping

In [ ]:
len(semantic_id_mapping)

In [ ]:
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

In [ ]:
dm.semantic_ids

In [ ]:
len(dm.semantic_ids)

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Semantic-ID diagnostics: pairwise code distances and per-codebook code frequencies.
# For large catalogs, pairwise distances are estimated from sampled item pairs to keep
# the analysis memory-safe.
semantic_codes = np.asarray(
    [semantic_id_mapping[item_id] for item_id in sorted(semantic_id_mapping)],
    dtype=np.int64,
)

if semantic_codes.ndim != 2 or semantic_codes.shape[0] == 0:
    raise ValueError("semantic_id_mapping must contain non-empty, fixed-length code lists.")

num_items, num_codebooks = semantic_codes.shape
max_pair_samples = 1_000_000
rng = np.random.default_rng(42)
num_all_pairs = num_items * (num_items - 1) // 2

if num_all_pairs <= max_pair_samples:
    pair_i, pair_j = np.triu_indices(num_items, k=1)
else:
    pair_i = rng.integers(0, num_items, size=max_pair_samples, endpoint=False)
    pair_j = rng.integers(0, num_items - 1, size=max_pair_samples, endpoint=False)
    pair_j = pair_j + (pair_j >= pair_i)

pairwise_hamming_distances = (semantic_codes[pair_i] != semantic_codes[pair_j]).sum(axis=1)
pairwise_exact_matches = (pairwise_hamming_distances == 0).sum()

print(f"Items: {num_items:,}")
print(f"Codebook levels: {num_codebooks:,}")
print(f"Pairwise distances computed: {len(pairwise_hamming_distances):,} of {num_all_pairs:,} possible pairs")
print(f"Exact semantic-code collisions in computed pairs: {pairwise_exact_matches:,}")
print(
    "Mean / median Hamming distance: "
    f"{pairwise_hamming_distances.mean():.3f} / {np.median(pairwise_hamming_distances):.3f}"
)

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(num_codebooks + 2) - 0.5
ax.hist(pairwise_hamming_distances, bins=bins, color="#4c78a8", edgecolor="white")
ax.set_xticks(range(num_codebooks + 1))
ax.set_xlabel("Pairwise Hamming distance between semantic codes")
ax.set_ylabel("Number of item pairs")
ax.set_title("Distribution of Pairwise Semantic-Code Distances")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

frequency_rows = []
for level in range(num_codebooks):
    counts = Counter(semantic_codes[:, level].tolist())
    for code, frequency in counts.items():
        frequency_rows.append(
            {
                "codebook_level": level,
                "code": code,
                "frequency": frequency,
                "frequency_share": frequency / num_items,
            }
        )

code_frequency_df = pd.DataFrame(frequency_rows)
display(
    code_frequency_df.groupby("codebook_level")["frequency"]
    .describe()
    .round(3)
)

ncols = min(3, num_codebooks)
nrows = math.ceil(num_codebooks / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

for level, ax in enumerate(axes.ravel()):
    if level >= num_codebooks:
        ax.axis("off")
        continue

    level_freqs = code_frequency_df.loc[
        code_frequency_df["codebook_level"] == level,
        "frequency",
    ]
    ax.hist(level_freqs, bins=40, color="#f58518", edgecolor="white")
    ax.set_title(f"Codebook Level {level}")
    ax.set_xlabel("Items assigned to code")
    ax.set_ylabel("Number of codes")
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Distribution of Code Frequencies per Codebook Level", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

split_datasets = {
    "train": dm.train_dataset,
    "val": dm.val_dataset,
    "test": dm.test_dataset,
}
subset_colors = {
    "train": "#4c78a8",
    "val": "#f58518",
    "test": "#54a24b",
}

if any(dataset is None for dataset in split_datasets.values()):
    raise RuntimeError("Call dm.setup() before plotting split token distributions.")

rows = []
for split_name, dataset in split_datasets.items():
    for example in dataset.examples:
        tokens = semantic_id_mapping.get(str(example.target_raw_item))
        if tokens is None:
            continue
        for level, token in enumerate(tokens):
            rows.append(
                {
                    "split": split_name,
                    "codebook_level": level,
                    "token": int(token),
                }
            )

split_token_df = pd.DataFrame(rows)
if split_token_df.empty:
    raise ValueError("No split target items matched semantic_id_mapping.")

num_levels = int(split_token_df["codebook_level"].max()) + 1
ncols = min(3, num_levels)
nrows = int(np.ceil(num_levels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.8 * nrows), squeeze=False)

for level, ax in enumerate(axes.ravel()):
    if level >= num_levels:
        ax.axis("off")
        continue

    level_df = split_token_df[split_token_df["codebook_level"] == level]
    token_min = int(level_df["token"].min())
    token_max = int(level_df["token"].max())


    for split_name in ("train", "val", "test"):
        split_tokens = level_df.loc[level_df["split"] == split_name, "token"].to_numpy()
        if len(split_tokens) == 0:
            continue
        weights = np.ones_like(split_tokens, dtype=float) / len(split_tokens)
        ax.hist(
            split_tokens,
            bins=50,
            weights=weights,
            color=subset_colors[split_name],
            alpha=0.45,
            label=split_name,
            edgecolor="none",
        )

    ax.set_title(f"Codebook Level {level}")
    ax.set_xlabel("Token")
    ax.set_ylabel("Normalized frequency")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)

fig.suptitle("Token Distribution Histograms by Split and Codebook Level", y=1.02)
plt.tight_layout()
plt.show()
